# RAG desde cero — dale a un LLM el conocimiento de TU empresa

Hoy montamos un RAG completo, en local y sin APIs de pago:

- **EmbeddingGemma** → convierte texto en vectores (el significado, hecho números)
- **ChromaDB** → base de datos vectorial (busca por significado en milisegundos)
- **Gemma 4** (`e2b`) → redacta la respuesta usando lo recuperado

SETUP

In [18]:
%pip install ollama chromadb numpy

Note: you may need to restart the kernel to use updated packages.


In [19]:
import ollama

_instalados = [m["model"] for m in ollama.list()["models"]] #modelos instalados
MODELO_EMBED = next((t for t in ["embeddinggemma", "embeddinggemma:300m", "embeddinggemma:latest"] if t in _instalados), "embeddinggemma") 

MODELO_CHAT = "qwen3:4b" #el modelo que redacta las respuestas, se puede cambiar por otro modelo de ollama

print("Embedding:", MODELO_EMBED)
print("Chat:", MODELO_CHAT)
print("Instalados:", MODELO_EMBED in _instalados, MODELO_CHAT in _instalados)

Embedding: embeddinggemma:300m
Chat: qwen3:4b
Instalados: True True


In [20]:
def preguntar(prompt, system=None):
    """Manda un mensaje al modelo de CHAT y devuelve solo el texto de la respuesta.
    prompt :  la pregunta o instrucción del usuario (rol "user").
    system : instrucciones de comportamiento opcionales para el modelo (rol "system").
        Aquí añadiremos más adelante las reglas del RAG.
    """

    # ollama espera una LISTA de mensajes con su rol (igual que la API de OPENAI)
    mensajes = []
    if system:
        mensajes.append({"role": "system", "content": system})
    mensajes.append({"role": "user", "content": prompt})

    opciones = {"temperature": 0} #opciones de generación, se pueden cambiar
    if "qwen" in MODELO_CHAT:
        opciones["think"] = False #qwen3 tiene una opción para "pensar" antes de responder, pero no siempre da buenos resultados

    r = ollama.chat(model=MODELO_CHAT, messages=mensajes, options=opciones)
    return r["message"]["content"].strip() #nos quedamos solo con el texto sin metadatos

print(preguntar("Di 'listo'"))

listo


Un LLM no puede responder ccosas que no sabe o no tiene en su base de conocmiento, como información específica de una empresa concreta.

In [21]:
pregunta_de_prueba = "¿Cuántos días de vacaciones me corresponden al año en Lumetra?"

print(preguntar(pregunta_de_prueba))

Lumetra no es un país ni una empresa reconocida en el contexto internacional. Por favor, verifica el nombre o proporciona más detalles para poder ayudarte. En muchos países y empresas, el número de días de vacaciones anuales varía (por ejemplo, en España son 20 días, en Francia 20 días, etc.), pero sin conocer el país o empresa específica, no puedo dar una respuesta precisa. 😊


EMBEDDINGS: Significado hecho números

In [22]:
r = ollama.embed(model=MODELO_EMBED, input="El gato duerme en el sofá")
vector = r["embeddings"][0] #el resultado es una lista de listas, nos quedamos con la primera

print(f"Dimensión del vector: {len(vector)}") 
print(f"Primeras 5 componentes: {vector[:5]}")

Dimensión del vector: 768
Primeras 5 componentes: [-0.131917, 0.03932223, 0.040686093, -0.022982778, 0.013355833]


In [23]:
import numpy as np

def coseno(a, b):
    """Calcula la similitud coseno entre dos vectores a y b."""
    a = np.array(a)
    b = np.array(b)
    # producto escalar (a @ b) dividido por el producto de las normas
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

frases = [
    "El gato duerme en el sofá",
    "Un felino descansa en el sillón",
    "La factura vence el 30 de junio",
    "El pago debe realizarse antes de fin de mes"
]

vecs = ollama.embed(model=MODELO_EMBED, input=frases)["embeddings"] #vectorizamos todas las frases


# comparamos todas las parejas posibles de frases
for i in range(len(frases)):
    for j in range(i+1, len(frases)):       
        print(f"{coseno(vecs[i], vecs[j]):.3f}  |  '{frases[i]}'  <->  '{frases[j]}'")


0.769  |  'El gato duerme en el sofá'  <->  'Un felino descansa en el sillón'
0.272  |  'El gato duerme en el sofá'  <->  'La factura vence el 30 de junio'
0.267  |  'El gato duerme en el sofá'  <->  'El pago debe realizarse antes de fin de mes'
0.227  |  'Un felino descansa en el sillón'  <->  'La factura vence el 30 de junio'
0.242  |  'Un felino descansa en el sillón'  <->  'El pago debe realizarse antes de fin de mes'
0.598  |  'La factura vence el 30 de junio'  <->  'El pago debe realizarse antes de fin de mes'


In [24]:
# Embeddinggemma se entrenó con "prefijos" que le indican qué papel juega el texto.
# Usar el prefijoi correcto mejora el emparejamiento pregunta <-> documento.

def embed_documentos(textos):
    """Vectoriza una LISTA de textos que vamos a guardar en la base (documentos)."""
    entrada = [f"title: none | text: {t}" for t in textos] #añadimos el prefijo (de documentos) "title: none | text:" a cada texto
    return ollama.embed(model=MODELO_EMBED, input=entrada)["embeddings"]

def embed_consulta(texto):
    """Vectoriza UNA pregunta que vamos a BUSCAR (consulta)."""
    entrada = f"task: search result | query: {texto}" #añadimos el prefijo (de consulta) "query:" a la pregunta
    return ollama.embed(model=MODELO_EMBED, input=entrada)["embeddings"][0] #devuelve un solo vector



Primer buscador semántico

In [25]:
comentarios = [
    "Me han cobrado dos veces la suscripción este mes",
    "La app va lentísima desde la última actualización",
    "No consigo restablecer mi contraseña, el correo no llega",
    "El cargo de mi tarjeta no corresponde con la tarifa contratada",
    "Me encanta el nuevo diseño del panel de control, ¡muy intuitivo!",
    "Quiero darme de baja y nadie me responde"
]

def buscar (consulta, textos, k=3):
    """Busca los k textos más relevantes para la consulta dada, usando similitud coseno.
    Es un TAG en miniatura: vectorizar todo -> comparar -> ordenar -> quedarnos con los mejores.    
    """
    vec_consulta = embed_consulta(consulta) #vectorizamos la consulta
    vecs_textos = embed_documentos(textos) #vectorizamos los textos (documentos)
    
    #calculamos la similitud coseno entre la consulta y cada texto
    puntuaciones = [coseno(vec_consulta, v) for v in vecs_textos]
    
    orden = sorted(range(len(textos)), key=lambda i: puntuaciones[i], reverse=True) #ordenamos los índices de los textos por su puntuación
    return [(textos[i], puntuaciones[i]) for i in orden[:k]] #devolvemos los k mejores textos con su puntuación 


for texto, p in buscar("problemas para pagar", comentarios):
    print(f"{p:.3f}  |  {texto}")


0.286  |  Me han cobrado dos veces la suscripción este mes
0.280  |  El cargo de mi tarjeta no corresponde con la tarifa contratada
0.231  |  No consigo restablecer mi contraseña, el correo no llega


In [26]:
from pathlib import Path

# Cargar todos los .txt de la carpeta datos/ en un diccionario {nombre: contenido}
docs = {}
for ruta in sorted(Path("datos").glob("*.txt")):
    docs[ruta.name] = ruta.read_text(encoding="utf-8")

for nombre, texto in docs.items():
    print(f"{nombre}: {len(texto)} caracteres")    

faq_soporte.txt: 1171 caracteres
manual_producto.txt: 1192 caracteres
onboarding.txt: 1142 caracteres
politica_teletrabajo.txt: 1172 caracteres
politica_vacaciones.txt: 1151 caracteres


CHUNKING

El chunking importa MÁS de lo que parece (y es la causa principal de un RAG malo). Un párrafo que mezcla 4 ideas produce un vector "promedio" que no se parece del todo a ninguna pregunta concreta -> ese chunk nunca gana la similitud.


In [27]:
# Troceamos cada documento en CHUNKS

chunks, metadatos = [], []
for nombre, texto in docs.items():
    for parrafo in texto.split("\n\n"): #los párrados por una línea en blanco
        parrafo = parrafo.strip() #quitamos espacios al principio y al final
        if len(parrafo) > 40: #si el párrafo tiene más de 40 caracteres, lo guardamos como chunk
            chunks.append(parrafo)
            metadatos.append({"fuente": nombre}) #guardamos el nombre del documento como metadato


print(f"{len(chunks)} chunks generados")   
print("Ejemplo de chunk:", chunks[0][:120],"...")         

23 chunks generados
Ejemplo de chunk: PREGUNTAS FRECUENTES DE SOPORTE — LUMETRA INSIGHT ...


Indexar en Chroma

Se guarda cada chunk con su id, texto, embedding y metadato con el archivo de origen.

In [28]:
import chromadb

# PersistentClient: los datos se guardan en disco y persisten entre ejecuciones
cliente = chromadb.PersistentClient(path="chroma_lumetra") 

# Una "colección" es como una tabla de base de datos donde guardaremos nuestros chunks y sus vectores
coleccion =  cliente.get_or_create_collection(
    "lumetras", metadata={"hnsw:space": "cosine"}
)


# upsert: inserta o actualiza un vector con su id, sus metadatos y su chunk de texto original
# Guardamos 4 cosas por chunk: id, texto, vector y metadato

coleccion.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))], #id único para cada chunk
    documents=chunks, #el texto original del chunk
    embeddings=embed_documentos(chunks), #el vector del chunk
    metadatas=metadatos #los metadatos (en este caso, la fuente)
)

print(f"Chunks guardados en la colección: {coleccion.count()}")

Chunks guardados en la colección: 23


In [29]:
def recuperar(pregunta, k=4):
    """Función de recuperación que busca los k chunks más relevantes para la pregunta dada.

     Usamos k=4 (y no 1) porque el chunk con el dato exacto no siempre es el #1 en la lista de resultados,
     a veces el modelo lo considera relevante pero no el más relevante.
     De esta forma, aunque el chunk con la respuesta exacta no sea el #1,
     al menos estará entre los 4 primeros y el modelo de CHAT podrá usarlo para redactar la respuesta.

    """
    #query_embeddings espera una lista de vectores de consulta, pero nosotros solo tenemos una pregunta, así que le pasamos una lista con un solo vector
    res = coleccion.query(query_embeddings=[embed_consulta(pregunta)], n_results=k)   
    #chroma devuelve listas de listas, nos quedamos con la primera (y única) consulta y hacemos zip para juntar cada chunk con su fuente 
    return list(zip(res["documents"][0], [m["fuente"] for m in res["metadatas"][0]])) 

for texto, fuente in recuperar("¿Cuántos días de vacaciones me corresponden?"):
    print(f"{fuente}: {texto[:110]}...")

politica_vacaciones.txt: Los días de vacaciones no disfrutados caducan el 31 de marzo del año siguiente y no son compensables económica...
politica_vacaciones.txt: Las vacaciones se solicitan a través de la aplicación interna Atlas RRHH (atlas.lumetra.es) con un mínimo de 1...
politica_vacaciones.txt: Todas las personas en plantilla de Lumetra disponen de 23 días laborables de vacaciones al año. Además, se sum...
politica_vacaciones.txt: Entre el 15 de julio y el 31 de agosto no se pueden disfrutar más de 10 días laborables seguidos, salvo autori...


Ejercicio

- 1.- Probad 3 preguntas sobre Lumetra, ¿sale el chunk correcto en la respuesta?
- 2.- Construir una preguntan con sinónimos.
- 3.- Intentad encontrar una pregunta donde el top 1 de chunk que sale esté equicovado

In [30]:
for texto, fuente in recuperar("¿me pagan algo por currar desde casa?"):
    print(f"{fuente}: {texto[:110]}...")

politica_teletrabajo.txt: Toda persona en modalidad híbrida recibe una ayuda de 35 euros mensuales en nómina en concepto de gastos de te...
politica_teletrabajo.txt: Durante julio y agosto se puede solicitar teletrabajo total (5 días remotos por semana). La solicitud se prese...
politica_teletrabajo.txt: El equipamiento adicional para el puesto remoto (silla ergonómica y monitor externo) se solicita una única vez...
politica_vacaciones.txt: Los días 24 y 31 de diciembre son no laborables por convenio interno de Lumetra y no descuentan días de vacaci...


In [31]:
SYSTEM_RAG ="""Eres el asistente interno de Lumetra.
Responde SOLO con la información del CONTEXTO.
Si la respuesta no está en el contexto, di exactamente: "No encuentro esa información en la documentación."
Cita siempre el documento del que sacas cada dato, entre corchetes:  [nombre_del_archivo]."""

def rag(pregunta, k=4, ver_contexto=False):
    """RAG completo = recuperar + generar respuesta."""

    trozos = recuperar(pregunta, k) #recuperamos los k chunks más relevantes para la pregunta

    contexto = "\n\n".join([f"[{fuente}]: {texto}" for texto, fuente in trozos]) #creamos un contexto con los chunks recuperados, indicando su fuente entre corchetes
    
    if ver_contexto:
        print("-" * 60, f"\n CONTEXTO RECUPERADO: \n\n{contexto}\n", "-" * 60)

    prompt = f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"
    return preguntar(prompt, system=SYSTEM_RAG) #mandamos el prompt al modelo de CHAT con las instrucciones del sistema    



In [32]:
# comparamos la MISMA pregunta con y sin RAG
print("--- SIN RAG ---")
print(preguntar(pregunta_de_prueba))
print("\n--- CON RAG ---")
print(rag("¿Cuántos días de vacaciones me corresponden al año en Lumetra?", ver_contexto=True))

--- SIN RAG ---
Lumetra no es un país ni una empresa reconocida. En muchos países, los empleados tienen alrededor de 20 días de vacaciones anuales, pero el número varía según el país y la empresa. Te recomiendo verificar las políticas laborales de tu país o la empresa en la que trabajas.

--- CON RAG ---
------------------------------------------------------------ 
 CONTEXTO RECUPERADO: 

[politica_vacaciones.txt]: Todas las personas en plantilla de Lumetra disponen de 23 días laborables de vacaciones al año. Además, se suma 1 día adicional por cada 3 años de antigüedad en la empresa, hasta un máximo de 3 días extra.

[politica_vacaciones.txt]: Los días 24 y 31 de diciembre son no laborables por convenio interno de Lumetra y no descuentan días de vacaciones. Los traslados de domicilio dan derecho a 2 días naturales adicionales, que también se solicitan por Atlas RRHH adjuntando justificante.

[politica_teletrabajo.txt]: Lumetra funciona con un modelo híbrido 3+2: tres días de teletraba

Probad el rag():

- 3 preguntas con respuesta -> presente en los documentos
- 1 pregunta sin respuesta
- 1 pregunta que cruce dos documentos
- ¿ Siempre cita las fuentes?

In [33]:
# SOLUCIÓN (batería de ejemplo) — preguntas variadas para estresar al asistente:
baterias = [
    "¿Qué días tengo que ir a la oficina obligatoriamente?",
    "¿Qué incluye el plan Pro y cuánto cuesta?",
    "¿Qué hago si no me llega el correo para resetear la contraseña?",
    "¿Cuál es la política de bajas por enfermedad?",   # ¡NO está en el corpus! → debe confesarlo
    "¿Puedo teletrabajar todo julio y cuántos días seguidos de vacaciones puedo coger en agosto?",  # cruza 2 docs
]
for p in baterias:
    print(f"\n❓ {p}")
    print(rag(p))


❓ ¿Qué días tengo que ir a la oficina obligatoriamente?
Los días presenciales obligatorios son martes y jueves [politica_teletrabajo.txt].

❓ ¿Qué incluye el plan Pro y cuánto cuesta?
El plan Pro cuesta 199 euros al mes e incluye 25 usuarios, dashboards ilimitados y acceso a la API. [manual_producto.txt]

❓ ¿Qué hago si no me llega el correo para resetear la contraseña?
Revisar la carpeta de spam y, si sigue sin aparecer, escribir a soporte indicando el correo de la cuenta. [faq_soporte.txt]

❓ ¿Cuál es la política de bajas por enfermedad?
No encuentro esa información en la documentación.

❓ ¿Puedo teletrabajar todo julio y cuántos días seguidos de vacaciones puedo coger en agosto?
No, no se puede trabajar desde casa todo julio (solo 5 días remotos por semana). En agosto, se pueden disfrutar máximo 10 días laborables seguidos de vacaciones. [politica_teletrabajo.txt] [politica_vacaciones.txt]


## Ejercicio:

**a) El efecto de `k`.** Lanza la misma pregunta con `k=1` y `k=8`. ¿Cuándo le falta información? ¿Cuándo le sobra ruido? (con `ver_contexto=True` se ve clarísimo).

**b) Tu propio documento.** Crea `datos/politica_formacion.txt` (invéntate la política: presupuesto anual, cómo se pide…). Re-ejecuta las celdas de chunking + indexado y pregúntale por ella. Acabas de "enseñarle" algo nuevo al sistema **sin tocar el modelo**.

**c) Mini-eval.** Pasa la batería de abajo con y sin RAG y cuenta aciertos. De esta forma puedes comparar modelos y configuraciones del RAG.

**d) (Si te sobra tiempo)** Cambia el chunking: trocea por tamaño fijo (p. ej. 300 caracteres) en vez de por párrafos y repite la mini-eval. ¿Mejora o empeora? ¿Por qué?


In [ ]:
preguntas_eval = [
    "¿Cuántos días de vacaciones anuales tengo?",
    "¿Qué días son obligatorios en oficina?",
    "¿Cuánto cuesta el plan Pro?",
    "¿En qué navegador falla la exportación a PDF?",
    "¿Qué curso es obligatorio la primera semana?",
    "¿Cuál es la política de bajas por enfermedad?",
]

# Para cada pregunta imprimimos las dos respuestas, recortadas a 150 caracteres:
for p in preguntas_eval:
    print(f"\n{'═' * 70}\n❓ {p}")
    print(f"  SIN RAG → {preguntar(p)[:150]}")    # sin contexto: falla o divaga
    print(f"  CON RAG → {rag(p)[:150]}")          # con contexto: acierta o confiesa

## Cierre

> **embeddings → BBDD vectorial → recuperar → prompt aumentado → generar (citando fuentes)**

Lo que le falta a esto para ser producción (temas para profundizar):

- **Chunking inteligente**: con solape, respetando secciones, tamaños adaptados al documento
- **Re-ranking**: un segundo modelo reordena los chunks recuperados antes de generar
- **Búsqueda híbrida**: combinar vectores con búsqueda clásica por palabras (BM25)
- **Evaluación automática**: baterías de preguntas/respuestas medidas con frameworks tipo RAGAS